# Model Training Demo

This notebook demonstrates the full training workflow:
1. Creating a synthetic IV surface tensor dataset
2. Instantiating the ConvLSTM forecast model
3. Running 2 training epochs
4. Plotting the training loss curve


In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# PyTorch imports
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import Dataset, DataLoader, random_split
    print(f'PyTorch version : {torch.__version__}')
    print(f'CUDA available  : {torch.cuda.is_available()}')
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device    : {DEVICE}')
except ImportError:
    print('PyTorch not installed. Run: pip install torch')
    raise

try:
    import pytorch_lightning as pl
    print(f'PyTorch Lightning: {pl.__version__}')
    PL_AVAILABLE = True
except ImportError:
    print('PyTorch Lightning not installed — using plain PyTorch training loop.')
    PL_AVAILABLE = False

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Synthetic IV surface tensor dataset
# ──────────────────────────────────────────────────────────────────────────────

class IVSurfaceDataset(Dataset):
    """
    Synthetic IV surface dataset for demonstration purposes.

    Generates N samples where each sample is a (lookback, C, H, W) tensor
    (sequence of surfaces) and the target is the next-day surface (C, H, W).

    Surface shape: C=1 channel, H=15 expiry nodes, W=25 strike nodes.
    Lookback: 30 days.
    """

    def __init__(
        self,
        n_samples: int = 500,
        lookback: int = 30,
        H: int = 15,
        W: int = 25,
        C: int = 1,
        seed: int = 42,
    ):
        self.n_samples = n_samples
        self.lookback = lookback
        self.H = H
        self.W = W
        self.C = C

        rng = np.random.default_rng(seed)

        # Simulate a correlated IV time series with mean-reversion
        total_steps = n_samples + lookback + 1
        atm_iv = np.zeros(total_steps)
        atm_iv[0] = 0.20
        kappa = 0.05   # mean-reversion speed
        mu    = 0.20   # long-run mean
        eta   = 0.02   # vol-of-vol
        dt    = 1/252
        for t in range(1, total_steps):
            atm_iv[t] = atm_iv[t-1] + kappa * (mu - atm_iv[t-1]) * dt + eta * np.sqrt(atm_iv[t-1] * dt) * rng.standard_normal()
            atm_iv[t] = np.clip(atm_iv[t], 0.05, 0.80)

        # Build surfaces: each surface is shaped (H, W)
        # Expiry dimension: term structure slope
        expiry_weights = np.linspace(1.0, 1.25, H).reshape(H, 1)  # upward slope
        # Strike dimension: smile (parabolic in log-moneyness)
        log_m = np.linspace(-0.30, 0.30, W)
        smile_weights = 1.0 - 0.5 * log_m + 2.0 * log_m**2   # skewed smile
        smile_weights = smile_weights.reshape(1, W)

        surfaces = np.zeros((total_steps, C, H, W), dtype=np.float32)
        for t in range(total_steps):
            base = atm_iv[t] * expiry_weights * smile_weights
            noise = rng.normal(0, 0.002, size=(H, W)).astype(np.float32)
            surfaces[t, 0] = (base + noise).astype(np.float32)

        # Build (X, y) pairs
        self.X = []  # shape (lookback, C, H, W)
        self.y = []  # shape (C, H, W)
        for i in range(n_samples):
            self.X.append(surfaces[i : i + lookback])          # (lookback, C, H, W)
            self.y.append(surfaces[i + lookback])              # (C, H, W) target

        self.X = np.array(self.X, dtype=np.float32)
        self.y = np.array(self.y, dtype=np.float32)

        print(f'Dataset created: X shape={self.X.shape}, y shape={self.y.shape}')
        print(f'IV range: [{self.X.min():.4f}, {self.X.max():.4f}]')

    def __len__(self) -> int:
        return self.n_samples

    def __getitem__(self, idx: int):
        return torch.from_numpy(self.X[idx]), torch.from_numpy(self.y[idx])


# Create dataset and dataloaders
dataset = IVSurfaceDataset(n_samples=500, lookback=30, H=15, W=25, C=1)

train_size = int(0.80 * len(dataset))
val_size   = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0, pin_memory=False)

print(f'\nTrain batches: {len(train_loader)}')
print(f'Val   batches: {len(val_loader)}')

# Inspect one batch
x_batch, y_batch = next(iter(train_loader))
print(f'\nBatch X shape: {x_batch.shape}   (batch, lookback, C, H, W)')
print(f'Batch y shape: {y_batch.shape}   (batch, C, H, W)')

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# ConvLSTM + CNN Forecast Model
# ──────────────────────────────────────────────────────────────────────────────

class ConvLSTMCell(nn.Module):
    """Single ConvLSTM cell."""

    def __init__(self, in_channels: int, hidden_channels: int, kernel_size: int = 3):
        super().__init__()
        pad = kernel_size // 2
        self.hidden_channels = hidden_channels
        # Combined gate convolution (input + hidden → 4 gates)
        self.conv = nn.Conv2d(
            in_channels + hidden_channels,
            4 * hidden_channels,
            kernel_size=kernel_size,
            padding=pad,
        )
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.conv.weight)
        nn.init.zeros_(self.conv.bias)

    def forward(self, x, h_prev, c_prev):
        combined = torch.cat([x, h_prev], dim=1)
        gates = self.conv(combined)
        i, f, g, o = gates.chunk(4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        g = torch.tanh(g)
        o = torch.sigmoid(o)
        c = f * c_prev + i * g
        h = o * torch.tanh(c)
        return h, c

    def init_hidden(self, batch_size: int, H: int, W: int, device):
        return (
            torch.zeros(batch_size, self.hidden_channels, H, W, device=device),
            torch.zeros(batch_size, self.hidden_channels, H, W, device=device),
        )


class IVSurfaceForecastModel(nn.Module):
    """
    ConvLSTM-based IV surface forecasting model.

    Architecture:
      Input  (B, T, C, H, W)
        → ConvLSTM encoder (2 layers)
        → Final hidden state (B, hidden, H, W)
        → CNN decoder (2 conv layers)
      Output (B, C, H, W) — next-day surface forecast
    """

    def __init__(
        self,
        in_channels: int = 1,
        hidden_channels: int = 32,
        n_lstm_layers: int = 2,
        out_channels: int = 1,
        kernel_size: int = 3,
    ):
        super().__init__()
        self.in_channels = in_channels
        self.hidden_channels = hidden_channels
        self.n_lstm_layers = n_lstm_layers

        # Build ConvLSTM layers
        self.conv_lstm_cells = nn.ModuleList()
        for layer in range(n_lstm_layers):
            cell_in_ch = in_channels if layer == 0 else hidden_channels
            self.conv_lstm_cells.append(
                ConvLSTMCell(cell_in_ch, hidden_channels, kernel_size)
            )

        # CNN decoder: hidden → output surface
        self.decoder = nn.Sequential(
            nn.Conv2d(hidden_channels, hidden_channels // 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden_channels // 2),
            nn.GELU(),
            nn.Conv2d(hidden_channels // 2, out_channels, kernel_size=1),
            nn.Softplus(),  # ensures non-negative IV output
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, C, H, W)"""
        B, T, C, H, W = x.shape
        device = x.device

        # Initialise hidden states
        hidden_states = [
            cell.init_hidden(B, H, W, device)
            for cell in self.conv_lstm_cells
        ]

        # Unroll through time steps
        for t in range(T):
            layer_input = x[:, t]  # (B, C, H, W)
            new_hidden = []
            for layer_idx, cell in enumerate(self.conv_lstm_cells):
                h, c = hidden_states[layer_idx]
                h_new, c_new = cell(layer_input, h, c)
                new_hidden.append((h_new, c_new))
                layer_input = h_new  # pass to next layer
            hidden_states = new_hidden

        # Decode final hidden state of last layer
        final_h = hidden_states[-1][0]  # (B, hidden, H, W)
        return self.decoder(final_h)    # (B, out_channels, H, W)


# Instantiate model
model = IVSurfaceForecastModel(
    in_channels=1,
    hidden_channels=32,
    n_lstm_layers=2,
    out_channels=1,
).to(DEVICE)

# Parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Model: IVSurfaceForecastModel (ConvLSTM)')
print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Device              : {DEVICE}')

# Forward pass sanity check
with torch.no_grad():
    dummy_x = torch.randn(4, 30, 1, 15, 25).to(DEVICE)   # batch=4, T=30, C=1, H=15, W=25
    dummy_y = model(dummy_x)
    print(f'\nForward pass check: input {dummy_x.shape} → output {dummy_y.shape}')

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Training loop — 2 epochs
# ──────────────────────────────────────────────────────────────────────────────

N_EPOCHS = 2

criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-3,
    steps_per_epoch=len(train_loader),
    epochs=N_EPOCHS,
    pct_start=0.3,
)

train_losses: list[float] = []
val_losses:   list[float] = []
step_losses:  list[float] = []

print(f'Starting training for {N_EPOCHS} epoch(s) on {DEVICE}...')
print('-' * 60)

for epoch in range(1, N_EPOCHS + 1):
    # ---- Training ----
    model.train()
    epoch_train_loss = 0.0
    for batch_idx, (x_batch, y_batch) in enumerate(train_loader):
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()
        y_pred = model(x_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()

        # Gradient clipping
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        batch_loss = loss.item()
        epoch_train_loss += batch_loss
        step_losses.append(batch_loss)

        if batch_idx % 5 == 0:
            current_lr = scheduler.get_last_lr()[0]
            print(f'  Epoch {epoch}/{N_EPOCHS}  Batch {batch_idx+1:3d}/{len(train_loader)}  '
                  f'loss={batch_loss:.6f}  lr={current_lr:.2e}')

    avg_train_loss = epoch_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # ---- Validation ----
    model.eval()
    epoch_val_loss = 0.0
    with torch.no_grad():
        for x_val, y_val in val_loader:
            x_val = x_val.to(DEVICE)
            y_val = y_val.to(DEVICE)
            y_pred_val = model(x_val)
            val_loss = criterion(y_pred_val, y_val)
            epoch_val_loss += val_loss.item()

    avg_val_loss = epoch_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    # Vol-point RMSE (approximate: multiply MSE by scale factor)
    val_rmse_vol = np.sqrt(avg_val_loss) * 100  # rough vol-point approximation

    print(f'\nEpoch {epoch}/{N_EPOCHS} Summary:')
    print(f'  Train Loss : {avg_train_loss:.6f}')
    print(f'  Val   Loss : {avg_val_loss:.6f}')
    print(f'  Val RMSE   : ~{val_rmse_vol:.2f} vol-pts')
    print('-' * 60)

print('Training complete.')

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Plot training loss curves
# ──────────────────────────────────────────────────────────────────────────────

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Step-level Training Loss', 'Epoch-level Train vs Val Loss'],
)

# Left: step-level loss
steps = list(range(1, len(step_losses) + 1))
# Smooth with rolling average
window = max(1, len(step_losses) // 20)
smoothed = pd.Series(step_losses).rolling(window=window, min_periods=1).mean().tolist()

fig.add_trace(
    go.Scatter(
        x=steps, y=step_losses,
        mode='lines',
        name='Raw loss',
        line=dict(color='lightblue', width=1),
        opacity=0.5,
    ),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=steps, y=smoothed,
        mode='lines',
        name='Smoothed loss',
        line=dict(color='royalblue', width=2.5),
    ),
    row=1, col=1
)

# Right: epoch-level train vs val
epochs = list(range(1, len(train_losses) + 1))
fig.add_trace(
    go.Scatter(
        x=epochs, y=train_losses,
        mode='lines+markers',
        name='Train loss',
        line=dict(color='royalblue', width=2),
        marker=dict(size=8),
    ),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(
        x=epochs, y=val_losses,
        mode='lines+markers',
        name='Val loss',
        line=dict(color='firebrick', width=2, dash='dash'),
        marker=dict(size=8, symbol='diamond'),
    ),
    row=1, col=2
)

fig.update_xaxes(title_text='Training Step', row=1, col=1)
fig.update_xaxes(title_text='Epoch', row=1, col=2)
fig.update_yaxes(title_text='MSE Loss', row=1, col=1)
fig.update_yaxes(title_text='MSE Loss', row=1, col=2)

fig.update_layout(
    title='IVSurfaceForecastModel — Training Loss Curves',
    template='plotly_white',
    width=1100,
    height=450,
    legend=dict(x=0.55, y=0.95),
)
fig.show()

print(f'\nFinal metrics after {N_EPOCHS} epoch(s):')
print(f'  Final train MSE : {train_losses[-1]:.6f}')
print(f'  Final val   MSE : {val_losses[-1]:.6f}')
print(f'  Final val RMSE  : ~{np.sqrt(val_losses[-1])*100:.2f} vol-pts')
print(f'\nNote: 2 epochs is for demo only. Production training uses 100+ epochs')
print(f'with walk-forward cross-validation and early stopping.')